In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. 라이브러리 설치 (이 줄이 제일 위에 있어야 함)
!pip install tiktoken

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import os
import requests
import time
from google.colab import drive

# 메모리 정리
import gc
torch.cuda.empty_cache()
gc.collect()

# --- [진단용] 로그 찍는 함수 ---
def print_log(step_name, tensor, detail=""):
    shape_str = f"{list(tensor.shape)}"
    print(f"[{step_name}] Shape: {shape_str} | {detail}")

# --- 2. 설정 ---
class GPTConfig:
    block_size = 256
    vocab_size = 100277
    n_layer = 12
    n_head = 8
    n_embd = 512
    dropout = 0.1
    batch_size = 16
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = GPTConfig()
print(f"현재 사용 장치: {config.device}")

# --- 3. 모델 정의 (X-Ray 기능 탑재) ---
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(config.n_embd, head_size, bias=False)
        self.query = nn.Linear(config.n_embd, head_size, bias=False)
        self.value = nn.Linear(config.n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(config.block_size, config.block_size)))
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, verbose=False, layer_idx=0):
        # 1. 어텐션
        x = x + self.sa(self.ln1(x))
        # 2. 피드포워드
        x = x + self.ffwd(self.ln2(x))

        # [X-Ray 핵심] verbose=True이고 첫 번째 레이어일 때만 로그 출력
        if verbose and layer_idx == 0:
             print_log(f"Layer {layer_idx} 완료", x, "블록 하나(Attn+FF) 통과 후 데이터 모양")
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)

        # [중요 변경] Sequential 대신 ModuleList 사용 (반복문 제어를 위해)
        self.blocks = nn.ModuleList([Block(config.n_embd, config.n_head) for _ in range(config.n_layer)])

        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # verbose 옵션 추가 (기본값 False)
    def forward(self, idx, targets=None, verbose=False):
        B, T = idx.shape

        if verbose:
            print(f"\n=== [X-Ray] 1. 입력 단계 ===")
            print_log("Input Tokens", idx, f"입력된 토큰 ID들")

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=config.device))
        x = tok_emb + pos_emb

        if verbose:
            print(f"\n=== [X-Ray] 2. 임베딩 단계 ===")
            print_log("Embeddings", x, f"토큰이 {config.n_embd}차원 벡터로 변환됨")

        # 블록 통과 (verbose 정보 전달)
        for i, block in enumerate(self.blocks):
            x = block(x, verbose=verbose, layer_idx=i)

        x = self.ln_f(x)

        if targets is None:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        else:
            logits = self.lm_head(x)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        if verbose:
            print(f"\n=== [X-Ray] 3. 최종 출력 단계 ===")
            print_log("Logits", logits, f"다음 단어 예측 점수 (Vocab Size: {config.vocab_size})")

        return logits, loss

현재 사용 장치: cuda


In [ ]:
# --- 4. 학습 준비 ---
max_iters = 3000
eval_interval = 200
learning_rate = 3e-4

# 데이터 준비
file_path = 'input.txt'
if not os.path.exists(file_path):
    print("데이터 다운로드 중...")
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(url).text)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

enc = tiktoken.get_encoding("cl100k_base")
data = torch.tensor(enc.encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - config.block_size, (config.batch_size,))
    x = torch.stack([data_source[i:i+config.block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+config.block_size+1] for i in ix])
    return x.to(config.device), y.to(config.device)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(50)
        for k in range(50):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                _, loss = model(X, Y) # verbose=False (기본값)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# --- 5. 학습 실행 ---
model = GPT()
model = model.to(config.device)
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler()

print(f"--- 학습 시작 (총 {max_iters} 단계) ---")
start_time = time.time()

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"[{iter}/{max_iters}] train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")

    xb, yb = get_batch('train')

    with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
        # [중요] 학습 때는 verbose를 안 씁니다 (조용히 학습)
        logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

end_time = time.time()
print(f"--- 학습 완료! (소요 시간: {(end_time - start_time)/60:.2f}분) ---")

모델 파라미터 수: 140.73M
--- 학습 시작 (총 3000 단계) ---


/tmp/ipython-input-125413904.py:51: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[0/3000] train loss: 11.6398, val loss: 11.6377
[200/3000] train loss: 5.4130, val loss: 5.9681
[400/3000] train loss: 4.5812, val loss: 5.6368
[600/3000] train loss: 4.0296, val loss: 5.6082
[800/3000] train loss: 3.5453, val loss: 5.6844
[1000/3000] train loss: 2.9574, val loss: 5.8841
[1200/3000] train loss: 2.2673, val loss: 6.2005
[1400/3000] train loss: 1.5985, val loss: 6.6242
[1600/3000] train loss: 0.9809, val loss: 7.0611
[1800/3000] train loss: 0.5837, val loss: 7.5823
[2000/3000] train loss: 0.3751, val loss: 7.8571
[2200/3000] train loss: 0.2686, val loss: 8.2372
[2400/3000] train loss: 0.2231, val loss: 8.4315
[2600/3000] train loss: 0.1874, val loss: 8.6220
[2800/3000] train loss: 0.1729, val loss: 8.8490
[2999/3000] train loss: 0.1544, val loss: 8.9247
--- 학습 완료! (소요 시간: 22.99분) ---


In [ ]:
# --- 6. 구글 드라이브 저장 ---
print("Google Drive 마운트 중...")
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/My_GPT_Models'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

save_path = os.path.join(save_dir, 'gpt2_small_custom.pth')
torch.save(model.state_dict(), save_path)
print(f"✅ 모델이 구글 드라이브에 저장되었습니다: {save_path}")

# --- 7. [X-Ray 모드] 내부 구조 확인 ---
print("\n" + "="*30)
print("   🚀 X-Ray 진단 모드 실행   ")
print("="*30)

# 테스트용 문장
test_sentence = "Hello AI"
test_tokens = enc.encode(test_sentence)
test_input = torch.tensor([test_tokens], dtype=torch.long).to(config.device)

model.eval() # 평가 모드로 전환
with torch.no_grad():
    # 여기서 verbose=True를 켜서 내부를 들여다봅니다!
    model(test_input, verbose=True)

print("\n✅ X-Ray 촬영 끝! 위 로그를 확인하세요.")

Google Drive 마운트 중...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 모델이 구글 드라이브에 저장되었습니다: /content/drive/MyDrive/My_GPT_Models/gpt2_small_custom.pth

   🚀 X-Ray 진단 모드 실행   

=== [X-Ray] 1. 입력 단계 ===
[Input Tokens] Shape: [1, 2] | 입력된 토큰 ID들

=== [X-Ray] 2. 임베딩 단계 ===
[Embeddings] Shape: [1, 2, 512] | 토큰이 512차원 벡터로 변환됨
[Layer 0 완료] Shape: [1, 2, 512] | 블록 하나(Attn+FF) 통과 후 데이터 모양

=== [X-Ray] 3. 최종 출력 단계 ===
[Logits] Shape: [1, 1, 100277] | 다음 단어 예측 점수 (Vocab Size: 100277)

✅ X-Ray 촬영 끝! 위 로그를 확인하세요.


In [ ]:
# 훈련 안 된 모델 (쌩으로 돌리기)


import torch
import torch.nn.functional as F
import tiktoken

# 1. 토크나이저 준비
enc = tiktoken.get_encoding("cl100k_base")

# 2. 문장 생성 함수 (Generate)
def generate(model, start_text, max_new_tokens=50, temperature=1.0):
    model.eval()
    # 입력 텍스트 -> 토큰 ID 변환
    start_ids = enc.encode(start_text)
    idx = torch.tensor(start_ids, dtype=torch.long, device=config.device).unsqueeze(0)

    print(f"입력: {start_text}")
    print("-" * 30)

    for _ in range(max_new_tokens):
        # context window(block_size)를 넘지 않게 자르기
        idx_cond = idx if idx.size(1) <= config.block_size else idx[:, -config.block_size:]

        # 모델 예측
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)

        # 샘플링
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return enc.decode(idx[0].tolist())

# --- 실행 부분 ---
print("🔵 [Scenario 1] 훈련되지 않은 Raw 모델 실행 중...")

# 모델 초기화 (랜덤 가중치)
raw_model = GPT().to(config.device)

# 생성 테스트
output = generate(raw_model, "Hello", max_new_tokens=30)
print(f"결과: {output}")

🔵 [Scenario 1] 훈련되지 않은 Raw 모델 실행 중...
입력: Hello
------------------------------
결과: Hellonings.with testData northeasterndm panorama Discussion dma (^_cols workshopcpp nb.readerarming()}bij belong—youewidthThumbnail My {

escaping Assistance Polygon isError School FUNCT cryptoc


In [ ]:
# 훈련된 모델 불러와서 돌리기

import os
from google.colab import drive

# --- 실행 부분 ---
print("🟠 [Scenario 2] 구글 드라이브에서 학습된 모델 불러오기...")

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 모델 초기화 및 가중치 로드
save_path = '/content/drive/MyDrive/My_GPT_Models/gpt2_small_custom.pth' # 저장했던 경로 확인

if os.path.exists(save_path):
    trained_model = GPT().to(config.device)

    # 저장된 가중치 덮어쓰기
    trained_model.load_state_dict(torch.load(save_path, map_location=config.device))
    print("✅ 모델 로드 성공!")

    # 3. 생성 테스트
    output = generate(trained_model, "who are you?", max_new_tokens=50) # 위에서 정의한 generate 함수 재사용
    print(f"결과: {output}")
else:
    print(f"❌ 파일이 없습니다. 경로를 확인해주세요: {save_path}")

🟠 [Scenario 2] 구글 드라이브에서 학습된 모델 불러오기...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 모델 로드 성공!
입력: who are you?
------------------------------
결과: who are you? 
This is only a friend of your father’s Harpurnia. 
she’s expression to her eyes to her navy voile dress and fussing him: 
She took her glasses and rendered her mouth together, when 
she saw her mouth
